# 02. Baseline Oversampling Methods
**GA-Optimized G-CTGAN: An Automated Oversampling Framework for Imbalanced Data Classification**

This notebook runs all **baseline oversampling methods** across all datasets and classifiers,
and saves results to `results/02_baselines_results.csv`.

| Method | Type | Reference |
|--------|------|-----------|
| None   | Baseline (no oversampling) | — |
| SMOTE  | Interpolation | Chawla et al., 2002 |
| ADASYN | Adaptive interpolation | He et al., 2008 |
| G-SMOTE | GMM + Interpolation | — |
| CTGAN  | GAN-based | Xu et al., NeurIPS 2019 |
| TVAE   | VAE-based | Xu et al., NeurIPS 2019 |

> **Note:** CTAB-GAN+ is handled separately in `03_ctabgan.ipynb` (requires `env_ctabgan` kernel).


## 0. Setup

In [2]:
import os
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────
DATASET_DIR = "./datasets"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Reproducibility ────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATASET_NAMES = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "yeast_me2", "mammography", "abalone_19", "wine_quality",
    "ecoli", "pageblocks", "protein_homo",
]

# TabNet removed — RF / LGBM / MLP only (consistent with Q-G-CTGAN experiment)
CLASSIFIERS = ["RF", "LGBM", "MLP"]

# G-CTGAN added as the immediate predecessor baseline
OVERSAMPLING_METHODS = ["None", "SMOTE", "ADASYN", "G-SMOTE", "CTGAN", "TVAE", "G-CTGAN"]

UNIFIED_RATIO = 0.50

print(f"Datasets    : {len(DATASET_NAMES)}")
print(f"Methods     : {OVERSAMPLING_METHODS}")
print(f"Classifiers : {CLASSIFIERS}")
print(f"Target ratio: {UNIFIED_RATIO}")


# ════════════════════════════════════════════════════════════
## 1. Classifiers  (identical hyperparameters to Q-G-CTGAN notebook)
# ════════════════════════════════════════════════════════════

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier


def get_classifier(name, random_state=RANDOM_STATE):
    if name == "RF":
        return RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        )
    elif name == "LGBM":
        return LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            class_weight="balanced", random_state=random_state,
            n_jobs=-1, verbose=-1,
        )
    elif name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), alpha=0.001,
            max_iter=300, random_state=random_state,
        )

print("Classifiers ready.")


# ════════════════════════════════════════════════════════════
## 2. Evaluation  (identical to Q-G-CTGAN notebook)
# ════════════════════════════════════════════════════════════

from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score


def evaluate(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    return {
        "AUC"      : round(roc_auc_score(y_test, y_prob), 4),
        "F1"       : round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Precision": round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Recall"   : round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4),
    }

print("Evaluation ready.")


# ════════════════════════════════════════════════════════════
## 3. Oversampling Functions
# ════════════════════════════════════════════════════════════

from sklearn.mixture import GaussianMixture
from imblearn.over_sampling import SMOTE, ADASYN
from ctgan import CTGAN, TVAE


def compute_n_samples(y_train, target_ratio=UNIFIED_RATIO):
    n_min = int((y_train == 1).sum())
    n_maj = int((y_train == 0).sum())
    n_target = int(n_maj * target_ratio / (1 - target_ratio))
    return max(0, n_target - n_min)


def apply_smote(X_train, y_train, random_state=RANDOM_STATE):
    sm = SMOTE(k_neighbors=3, random_state=random_state)
    return sm.fit_resample(X_train, y_train)


def apply_adasyn(X_train, y_train, random_state=RANDOM_STATE):
    ad = ADASYN(random_state=random_state)
    return ad.fit_resample(X_train, y_train)


def apply_gsmote(X_train, y_train, random_state=RANDOM_STATE):
    """G-SMOTE: GMM clustering on minority class, then SMOTE per cluster."""
    X_min = X_train[y_train == 1]
    X_maj = X_train[y_train == 0]

    bic_scores = {}
    for k in range(2, min(11, len(X_min) // 2)):
        try:
            gmm = GaussianMixture(n_components=k, random_state=random_state)
            gmm.fit(X_min)
            bic_scores[k] = gmm.bic(X_min)
        except Exception:
            continue
    if not bic_scores:
        return apply_smote(X_train, y_train, random_state)

    best_k = min(bic_scores, key=bic_scores.get)
    gmm    = GaussianMixture(n_components=best_k, random_state=random_state)
    gmm.fit(X_min)
    labels = gmm.predict(X_min)

    X_syn_list = []
    for c in range(best_k):
        X_c = X_min[labels == c]
        if len(X_c) < 2:
            continue
        X_comb = np.vstack([X_maj, X_c])
        y_comb = np.array([0] * len(X_maj) + [1] * len(X_c))
        try:
            sm = SMOTE(k_neighbors=min(3, len(X_c) - 1), random_state=random_state)
            X_res, y_res = sm.fit_resample(X_comb, y_comb)
            X_syn_list.append(X_res[len(X_comb):])
        except Exception:
            continue

    if not X_syn_list:
        return apply_smote(X_train, y_train, random_state)

    X_syn = np.vstack(X_syn_list)
    X_out = np.vstack([X_train, X_syn])
    y_out = np.concatenate([y_train, np.ones(len(X_syn), dtype=int)])
    return X_out, y_out


def apply_ctgan(X_train, y_train, random_state=RANDOM_STATE):
    """CTGAN: train on minority class, sample n_synthetic rows."""
    n_syn = compute_n_samples(y_train)
    if n_syn == 0:
        return X_train, y_train
    X_min    = X_train[y_train == 1].astype(float)
    cols     = [f"f{i}" for i in range(X_min.shape[1])]
    X_min_df = pd.DataFrame(X_min, columns=cols)
    model    = CTGAN(epochs=100, verbose=False)
    model.fit(X_min_df)
    X_syn = model.sample(n_syn).values.astype(float)
    X_out = np.vstack([X_train.astype(float), X_syn])
    y_out = np.concatenate([y_train, np.ones(n_syn, dtype=int)])
    return X_out, y_out


def apply_tvae(X_train, y_train, random_state=RANDOM_STATE):
    """TVAE: train on minority class, sample n_synthetic rows."""
    n_syn = compute_n_samples(y_train)
    if n_syn == 0:
        return X_train, y_train
    X_min    = X_train[y_train == 1].astype(float)
    cols     = [f"f{i}" for i in range(X_min.shape[1])]
    X_min_df = pd.DataFrame(X_min, columns=cols)
    model    = TVAE(epochs=100)
    model.fit(X_min_df)
    X_syn = model.sample(n_syn).values.astype(float)
    X_out = np.vstack([X_train.astype(float), X_syn])
    y_out = np.concatenate([y_train, np.ones(n_syn, dtype=int)])
    return X_out, y_out


def apply_gctgan(X_train, y_train, random_state=RANDOM_STATE):
    """
    G-CTGAN: GMM clustering on minority class (BIC-optimal k),
    train independent CTGAN per cluster, synthesise proportionally.
    Fixed total ratio = UNIFIED_RATIO (no GA optimisation).
    """
    X_min = X_train[y_train == 1].astype(float)
    X_maj = X_train[y_train == 0].astype(float)
    n_syn_total = compute_n_samples(y_train)
    if n_syn_total == 0:
        return X_train, y_train

    # BIC-optimal k selection
    bic_scores = {}
    for k in range(2, min(11, len(X_min) // 2)):
        try:
            gmm = GaussianMixture(
                n_components=k, covariance_type="full",
                random_state=random_state, max_iter=200,
            )
            gmm.fit(X_min)
            bic_scores[k] = gmm.bic(X_min)
        except Exception:
            continue
    if not bic_scores:
        return apply_ctgan(X_train, y_train, random_state)

    best_k = min(bic_scores, key=bic_scores.get)
    gmm    = GaussianMixture(
        n_components=best_k, covariance_type="full",
        random_state=random_state, max_iter=200,
    )
    gmm.fit(X_min)
    labels        = gmm.predict(X_min)
    cluster_sizes = np.bincount(labels)

    # Allocate synthetic samples proportionally to cluster size
    X_syn_list = []
    for c in range(best_k):
        X_c = X_min[labels == c]
        if len(X_c) < 2:
            continue
        n_syn_c  = max(1, int(np.round(n_syn_total * len(X_c) / len(X_min))))
        cols     = [f"f{i}" for i in range(X_c.shape[1])]
        X_c_df   = pd.DataFrame(X_c, columns=cols)
        try:
            model = CTGAN(epochs=100, verbose=False)
            model.fit(X_c_df)
            X_syn_list.append(model.sample(n_syn_c).values.astype(float))
        except Exception:
            continue

    if not X_syn_list:
        return apply_ctgan(X_train, y_train, random_state)

    X_syn = np.vstack(X_syn_list)
    X_out = np.vstack([X_maj, X_min, X_syn])
    y_out = np.concatenate([
        np.zeros(len(X_maj)), np.ones(len(X_min)), np.ones(len(X_syn))
    ]).astype(int)
    return X_out, y_out


OVERSAMPLERS = {
    "None"   : None,
    "SMOTE"  : apply_smote,
    "ADASYN" : apply_adasyn,
    "G-SMOTE": apply_gsmote,
    "CTGAN"  : apply_ctgan,
    "TVAE"   : apply_tvae,
    "G-CTGAN": apply_gctgan,
}

print("Oversampling functions ready.")


# ════════════════════════════════════════════════════════════
## 4. Main Experiment Loop
# ════════════════════════════════════════════════════════════

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

results = []
TOTAL   = len(DATASET_NAMES) * len(OVERSAMPLING_METHODS) * len(CLASSIFIERS)
done    = 0

for ds_name in DATASET_NAMES:
    path = os.path.join(DATASET_DIR, f"{ds_name}.csv")
    if not os.path.exists(path):
        print(f"[SKIP] {ds_name}")
        done += len(OVERSAMPLING_METHODS) * len(CLASSIFIERS)
        continue

    df = pd.read_csv(path)
    X  = df.drop(columns=["target"]).values.astype(float)
    y  = df["target"].values.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
    )
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    print(f"\n{'='*60}")
    print(f"Dataset : {ds_name}  |  n_train={len(X_train):,}  |  "
          f"minority={y_train.mean():.2%}")
    print(f"{'='*60}")

    for method in OVERSAMPLING_METHODS:
        t0 = time.time()
        try:
            if method == "None":
                X_res, y_res = X_train.copy(), y_train.copy()
            else:
                X_res, y_res = OVERSAMPLERS[method](X_train.copy(), y_train.copy())
            oversample_time = round(time.time() - t0, 2)
        except Exception as e:
            print(f"  [{method}] Oversampling failed: {e}")
            done += len(CLASSIFIERS)
            continue

        ratio_res = round((y_res == 1).sum() / len(y_res), 4)

        for clf_name in CLASSIFIERS:
            done += 1
            progress = f"[{done:>4}/{TOTAL}]"
            clf = get_classifier(clf_name)
            t1  = time.time()
            try:
                clf.fit(X_res, y_res)
                train_time = round(time.time() - t1, 2)
                metrics    = evaluate(clf, X_test, y_test)
            except Exception as e:
                print(f"  {progress} {ds_name} | {method:<8} | {clf_name} — Failed: {e}")
                continue

            results.append({
                "dataset"        : ds_name,
                "oversampling"   : method,
                "classifier"     : clf_name,
                "n_resampled"    : len(y_res),
                "minority_ratio" : ratio_res,
                "oversample_time": oversample_time,
                "train_time"     : train_time,
                **metrics,
            })
            print(f"  {progress} {ds_name} | {method:<8} | {clf_name:<5} | "
                  f"AUC={metrics['AUC']:.4f}  F1={metrics['F1']:.4f}  "
                  f"[{train_time:.1f}s]")

print("\nExperiment complete.")


# ════════════════════════════════════════════════════════════
## 5. Save & Summary
# ════════════════════════════════════════════════════════════

results_df = pd.DataFrame(results)
save_path  = os.path.join(RESULTS_DIR, "02_baselines_results.csv")
results_df.to_csv(save_path, index=False)
print(f"Results saved → {save_path}")

print("\nMean AUC by method & classifier:")
print(results_df.groupby(["oversampling", "classifier"])["AUC"]
      .mean().round(4).unstack().to_string())


# ════════════════════════════════════════════════════════════
## 6. Combined Comparison Table (Baselines + Q-G-CTGAN)
# ════════════════════════════════════════════════════════════

qg_path = os.path.join(RESULTS_DIR, "03_qgctgan_results.csv")
if os.path.exists(qg_path):
    qg_df = pd.read_csv(qg_path)

    # Keep only adaptive α as the representative Q-G-CTGAN result
    qg_best = qg_df[qg_df["oversampling"] == "Q-G-CTGAN_adaptive"].copy()

    combined = pd.concat([results_df, qg_best], ignore_index=True)

    METHOD_ORDER = [
        "None", "SMOTE", "ADASYN", "G-SMOTE",
        "CTGAN", "TVAE", "G-CTGAN", "Q-G-CTGAN_adaptive"
    ]

    pivot_auc = (
        combined.groupby(["oversampling", "classifier"])["AUC"]
        .mean().round(4).unstack()
        .reindex([m for m in METHOD_ORDER if m in combined["oversampling"].unique()])
    )

    pivot_f1 = (
        combined.groupby(["oversampling", "classifier"])["F1"]
        .mean().round(4).unstack()
        .reindex([m for m in METHOD_ORDER if m in combined["oversampling"].unique()])
    )

    print("\n── Mean AUC (all datasets) ──")
    print(pivot_auc.to_string())
    print("\n── Mean F1  (all datasets) ──")
    print(pivot_f1.to_string())

    # Save combined
    combined_path = os.path.join(RESULTS_DIR, "04_combined_comparison.csv")
    combined.to_csv(combined_path, index=False)
    print(f"\nCombined results saved → {combined_path}")
else:
    print("Q-G-CTGAN results not found. Run notebook 06 first.")

Datasets    : 11
Methods     : ['None', 'SMOTE', 'ADASYN', 'G-SMOTE', 'CTGAN', 'TVAE', 'G-CTGAN']
Classifiers : ['RF', 'LGBM', 'MLP']
Target ratio: 0.5
Classifiers ready.
Evaluation ready.
Oversampling functions ready.

Dataset : credit_default  |  n_train=21,000  |  minority=22.12%
  [   1/231] credit_default | None     | RF    | AUC=0.7757  F1=0.7023  [0.5s]
  [   2/231] credit_default | None     | LGBM  | AUC=0.7787  F1=0.6858  [0.1s]
  [   3/231] credit_default | None     | MLP   | AUC=0.6961  F1=0.6267  [50.7s]
  [   4/231] credit_default | SMOTE    | RF    | AUC=0.7733  F1=0.6930  [0.9s]
  [   5/231] credit_default | SMOTE    | LGBM  | AUC=0.7719  F1=0.6937  [0.1s]
  [   6/231] credit_default | SMOTE    | MLP   | AUC=0.6901  F1=0.6293  [77.4s]
  [   7/231] credit_default | ADASYN   | RF    | AUC=0.7684  F1=0.6861  [1.0s]
  [   8/231] credit_default | ADASYN   | LGBM  | AUC=0.7680  F1=0.6928  [0.2s]
  [   9/231] credit_default | ADASYN   | MLP   | AUC=0.6788  F1=0.6121  [82.5s]
  